# Chapter 5 — becoming a backprop ninja

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 5 — becoming a backprop ninja

**Video:** 1h56m · [youtu.be/q8SA3rM6ckI](https://youtu.be/q8SA3rM6ckI) · **Format:** an exercise, not a lecture. **The hardest chapter in the course**, by broad agreement among people who have done it.

### The problem

Chapter 1 built backpropagation for single numbers. Real networks propagate gradients through whole tensors, where the bookkeeping gets tricky: broadcasting, sums over dimensions, index lookups, and matrix multiplies each need their own backward rule. `loss.backward()` hides all of it.

This chapter deletes `loss.backward()` and makes you do the entire backward pass by hand, through cross-entropy, a linear layer, `tanh`, BatchNorm, another linear layer, and back into the embedding table. Karpathy notes this used to be the job: "about 10 years ago in deep learning this was fairly standard and in fact pervasive, at the time everyone used to write their own backward pass" [transcript].

> **Say it to a six-year-old.** You have a machine with a hundred pipes and water flowing backwards through it. At every join you have to work out how much water goes down each pipe. It is not hard at any single join. It is hard because there are a hundred of them and if you get one wrong, the water comes out in the wrong place and nothing tells you.

### The three rules that make tensor backprop tractable

**Rule 1: forward broadcasting becomes backward summing.**

If a bias vector of 6 numbers is added to a 4×6 matrix, that vector was silently copied onto 4 rows [transcript]. Each copy contributed separately to the loss, so the bias's gradient is the sum of the gradients of all 4 copies.

**Run it.**

In [ ]:
import torch
x = torch.randn(4, 6, requires_grad=True)
b = torch.randn(6, requires_grad=True)
(x + b).sum().backward()
print("b.grad:", [round(v, 3) for v in b.grad.tolist()])

**What you should see:**

**Expected output:**

```
b.grad: [4.0, 4.0, 4.0, 4.0, 4.0, 4.0]
```

[verified]

Every entry is 4.0, which is exactly the number of rows the bias was copied onto. **Wherever the forward pass duplicated something, the backward pass adds up.**

**Rule 2: forward indexing becomes backward scattering, with accumulation.**

The embedding lookup `C[idx]` picks rows out of a table. Going backward, each gradient must be deposited into the row it came from, using `+=` rather than `=`, "because there could be multiple" occurrences of the same character in a batch [transcript].

**Run it.**

In [ ]:
C = torch.randn(5, 3, requires_grad=True)
idx = torch.tensor([0, 2, 0, 0])       # row 0 is used three times
C[idx].sum().backward()
print("gradient totals per row:", C.grad.sum(1).tolist())

**What you should see:**

**Expected output:**

```
gradient totals per row: [9.0, 0.0, 3.0, 0.0, 0.0]
```

[verified]

Row 0 got 9.0 because it was used three times and each use contributes 3 (one per column). Row 2 got 3.0 from its single use. Rows 1, 3, 4 were never looked up and correctly got nothing. This is Chapter 1's accumulation rule wearing tensor clothing.

**Rule 3: the softmax-and-cross-entropy gradient collapses into something beautiful.**

The full derivative of softmax composed with cross-entropy looks fearsome. Worked through, it reduces to: **the predicted probabilities, minus 1 at the position of the correct answer, divided by the batch size.**

**Run it.** Do not take this on faith; check it against PyTorch:

In [ ]:
import torch.nn.functional as F
torch.manual_seed(42)
n, V = 4, 6
logits = torch.randn(n, V, requires_grad=True)
Y = torch.tensor([1, 3, 0, 5])
F.cross_entropy(logits, Y).backward()

probs = F.softmax(logits, dim=1)
manual = probs.clone()
manual[range(n), Y] -= 1     # subtract 1 at the correct answer
manual /= n                  # because the loss is a mean over n examples

print("PyTorch dlogits row 0:", [round(v, 5) for v in logits.grad[0].tolist()])
print("manual  dlogits row 0:", [round(v, 5) for v in manual[0].tolist()])
print("exact match:", torch.allclose(logits.grad, manual))

**What you should see:**

**Expected output:**

```
PyTorch dlogits row 0: [0.1064, -0.18145, 0.03813, 0.00189, 0.03053, 0.00451]
manual  dlogits row 0: [0.1064, -0.18145, 0.03813, 0.00189, 0.03053, 0.00451]
exact match: True
```

[verified]

**Sit with that result, because it is the most intuitive fact in the course.** The gradient on the outputs is "what you predicted minus what was true." Every wrong class gets a positive gradient, meaning push this score down. The correct class gets a negative one, meaning push this score up. The size of the push is exactly how wrong you were. If you had assigned the truth 100%, the gradient would be zero and nothing would change.

Karpathy calls the result "beautiful and very simple" and visualizes it as a grid of 32 examples by 27 characters, black squares marking the correct answers being pulled up while everything else is pushed down. [transcript]

**Why divide by n.** The loss is the *mean* over examples, and the derivative of `(a+b+c+d)/4` with respect to `a` is `1/4`. Forget this and every gradient is 4× too large, which silently multiplies your learning rate by the batch size. This exact mistake reappears in Chapter 9 with gradient accumulation.

### Why bother, if PyTorch does this for you

Abstractions leak, and this chapter is about the leaks:

- **Vanishing and exploding gradients.** Gradients that repeatedly shrink through many layers stop the early layers from learning; gradients that grow make training diverge. Both are chains of multiplications, and you cannot reason about them without a feel for what multiplies with what.
- **Gradient clipping quietly discards data.** Karpathy's observation: clipping a large gradient means "you are setting its gradient to zero," so outlier examples get ignored rather than dampened [transcript]. That is a modelling decision disguised as a numerical safeguard.
- **Dead neurons** (Chapter 4) are a gradient-flow phenomenon, and now you can see exactly where the flow stops: `1 − t²` is near zero, so everything upstream is multiplied by nearly nothing.

### Exercises

This chapter *is* the exercise. Karpathy's instruction is to attempt each gradient yourself and unpause only when stuck.

1. **Do rules 1 to 3 by hand** for a 2-layer network without looking, then check every tensor with `torch.allclose` against the autograd result. That comparison loop is the whole method.
2. **Derive the `tanh` backward rule** from the value alone: given `t = tanh(x)`, show the local slope is `1 − t²`. Verify by nudging.
3. **Backprop through BatchNorm.** This is the genuinely hard one, because the mean and standard deviation both depend on every example in the batch, so each example's gradient flows back through every other example's contribution. Expect it to take an hour.
4. **Break one gradient on purpose**, for instance drop the `/n`, and train. Note that the model still trains, just worse. That is why these bugs survive in real code.

### Troubleshooting

| Symptom | Cause |
|---|---|
| Your manual gradient is off by exactly the batch size | Missing the `/n` from the mean |
| Off by a constant factor of 2 | A squared term's chain rule dropped its factor of 2 |
| Correct for most entries, wrong for a few | Missing accumulation on a reused variable (rule 2) |
| Shapes match but values are transposed nonsense | A matrix multiply backward needs the *transpose*: for `Y = X @ W`, `dX = dY @ W.T` and `dW = X.T @ dY` |
| `torch.allclose` fails at 1e-8 | That is floating-point noise, not an error; compare with `atol=1e-6` |

> **For the PhD in the room.** The softmax-cross-entropy result is the standard exponential-family identity: for a log-partition function A(η), ∇A = E[T(x)], so the gradient of the negative log likelihood is the difference between expected and observed sufficient statistics, here p − y. The same identity is why logistic regression, softmax regression, and every generalized linear model share the "prediction minus target" gradient form. Practically, this is also why fusing softmax with cross-entropy is not merely a speed optimization: computing them separately means materializing log(p) where p may have underflowed, whereas the fused form is evaluated in logit space via the log-sum-exp trick and is stable.

### 30-second version

Delete the automatic differentiation and compute every gradient in a two-layer network by hand, including the awkward ones through batch normalization. Three rules cover most of it: forward duplication becomes backward summation, forward indexing becomes backward scattering with accumulation, and the gradient at a classifier's output is exactly "predicted probabilities minus the truth, divided by the batch size." You will never do this at work, and it is the fastest way to stop finding gradients mysterious.

---